In [1]:

import os

In [2]:
%pwd

'/home/caleb/projects/End-to-End-Chest-Cancer-Classification-using-MLflow-DVC/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/home/caleb/projects/End-to-End-Chest-Cancer-Classification-using-MLflow-DVC'

In [5]:
# import dagshub
# dagshub.init(repo_owner='calebomariba', repo_name='End-to-End-Chest-Cancer-Classification-using-MLflow-DVC', mlflow=True)

# import mlflow
# with mlflow.start_run():
#   mlflow.log_param('parameter name', 'value')
#   mlflow.log_metric('metric name', 1)

In [6]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/calebomariba/End-to-End-Chest-Cancer-Classification-using-MLflow-DVC.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="calebomariba"
os.environ["MLFLOW_TRACKING_PASSWORD"]="d7c4bd0848fc3e867145bbed38c08148a0d44e2b"

In [7]:
# https://dagshub.com/calebomariba/End-to-End-Chest-Cancer-Classification-using-MLflow-DVC.mlflow

In [8]:
import tensorflow as tf

2026-06-19 11:23:03.825748: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-19 11:23:03.829155: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-19 11:23:03.927273: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-19 11:23:03.975116: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-19 11:23:04.077771: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

In [9]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [10]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [11]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [ ]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model=Path("artifacts/training/model.h5"),
            training_data=Path("artifacts/data_ingestion/Chest-CT-Scan-data"),
            mlflow_uri="https://dagshub.com/calebomariba/End-to-End-Chest-Cancer-Classification-using-MLflow-DVC.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [13]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [14]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )
            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model, "model")

In [15]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

Found 102 images belonging to 2 classes.
7/7 ━━━━━━━━━━━━━━━━━━━━ 21s 3s/step - accuracy: 0.8627 - loss: 0.2121


2026/06/19 11:23:39 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/06/19 11:26:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 2
Created version '2' of model 'VGG16Model'.


In [16]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("https://dagshub.com/calebomariba/End-to-End-Chest-Cancer-Classification-using-MLflow-DVC.mlflow")

client = MlflowClient()

# Step 1 — List all versions of your registered model
model_name = "VGG16Model"
versions = client.search_model_versions(f"name='{model_name}'")
for v in versions:
    print(f"Version: {v.version} | Stage: {v.current_stage} | Run ID: {v.run_id}")

Version: 2 | Stage: None | Run ID: 9d5c08caa333452a9742b8d758c3b715
Version: 1 | Stage: None | Run ID: 25a68bfdf5e44dab8760b67c231613d9


In [17]:
# Step 2 — Delete specific model versions
client.delete_model_version(name="VGG16Model", version=1)
client.delete_model_version(name="VGG16Model", version=2)

# Or delete the entire registered model (all versions at once)
client.delete_registered_model(name="VGG16Model")

In [18]:
# Step 3 — Now delete the runs
runs = client.search_runs(experiment_ids=["0"])
for run in runs:
    print(f"Deleting: {run.info.run_id} | {run.info.run_name}")
    client.delete_run(run.info.run_id)

Deleting: 9d5c08caa333452a9742b8d758c3b715 | flawless-mouse-370
Deleting: 25a68bfdf5e44dab8760b67c231613d9 | efficient-stork-224


In [20]:
import os
import mlflow
from mlflow.tracking import MlflowClient

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/calebomariba/End-to-End-Chest-Cancer-Classification-using-MLflow-DVC.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "calebomariba"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "d7c4bd0848fc3e867145bbed38c08148a0d44e2b"

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
client = MlflowClient()

client.transition_model_version_stage(
    name="VGG16Model",
    version=3,
    stage="Production"
)
print("Model transitioned to Production")

/tmp/ipykernel_4540/3777458421.py:12: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.12.2/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


Model transitioned to Production
